## Desafio 1 , NLP , Tatiana Arenas Suárez

**1**. Vectorizar documentos. Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2**. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación
(f1-score macro) en el conjunto de datos de test. Considerar cambiar parámteros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial
y ComplementNB.

**3**. Transponer la matriz documento-término. De esa manera se obtiene una matriz
término-documento que puede ser interpretada como una colección de vectorización de palabras.
Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares. **La elección de palabras no debe ser al azar para evitar la aparición de términos poco interpretables, elegirlas "manualmente"**.


**1**. Vectorizar documentos. Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

In [19]:
# se cargan las librerias necesarias 
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import Pipeline
from pprint import pprint
import numpy as np 
import random

In [ ]:
# cargamos los datos (ya separados de forma predeterminada en train y test)
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

# se instancia el vectorizador
tfidfvect = TfidfVectorizer()
# se separa el target de los datos 
X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target
# se muestran las equivalencias 
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}
for k,v in tfidfvect.vocabulary_.items():
    print(v, k)

95844 was
97181 wondering
48754 if
18915 anyone
68847 out
88638 there
30074 could
37335 enlighten
60560 me
68080 on
88767 this
25775 car
80623 saw
88532 the
68781 other
31990 day
51326 it
34809 door
84538 sports
57390 looked
89360 to
21987 be
41715 from
55746 late
9843 60s
35974 early
11174 70s
25492 called
24160 bricklin
34810 doors
96247 were
76471 really
83426 small
49447 in
16809 addition
41724 front
24635 bumper
81658 separate
77878 rest
67670 of
23480 body
51136 is
17936 all
54632 know
25590 can
88143 tellme
62746 model
64931 name
37287 engine
84276 specs
99911 years
73373 production
96433 where
59079 made
46814 history
68409 or
96395 whatever
49932 info
100208 you
45885 have
41979 funky
57393 looking
71850 please
59216 mail
39384 fair
66857 number
24025 brave
84005 souls
96532 who
92659 upgraded
88565 their
82550 si
27947 clock
68705 oscillator
82058 shared
38725 experiences
41127 for
72238 poll
81586 send
24177 brief
61072 message
33193 detailing
100221 your
96917 with
73321 pr

In [5]:
# Se generan numeros al azar que determinene los documentos para comparar
numeros = [random.randint(0, 11314) for _ in range(5)]
numeros

[3607, 1064, 2434, 8765, 7166]

In [8]:
for i in numeros:
 print(f" --- Documento indice {i}, categoría: {newsgroups_train.target_names[y_train[i]]}---")
 cossim = cosine_similarity(X_train[i], X_train)[0]
 print("Los documentos más similares pertenecen a las categorías:")
 mostsim = np.argsort(cossim)[::-1][1:6]
 for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])


 --- Documento indice 3607, categoría: alt.atheism---
Los documentos más similares pertenecen a las categorías:
alt.atheism
sci.crypt
sci.crypt
alt.atheism
talk.politics.mideast
 --- Documento indice 1064, categoría: rec.sport.baseball---
Los documentos más similares pertenecen a las categorías:
rec.sport.baseball
rec.sport.baseball
rec.sport.baseball
rec.sport.baseball
rec.sport.baseball
 --- Documento indice 2434, categoría: rec.autos---
Los documentos más similares pertenecen a las categorías:
rec.motorcycles
talk.politics.mideast
sci.electronics
soc.religion.christian
talk.religion.misc
 --- Documento indice 8765, categoría: comp.sys.ibm.pc.hardware---
Los documentos más similares pertenecen a las categorías:
comp.sys.ibm.pc.hardware
comp.sys.ibm.pc.hardware
comp.sys.ibm.pc.hardware
comp.sys.ibm.pc.hardware
comp.sys.ibm.pc.hardware
 --- Documento indice 7166, categoría: rec.motorcycles---
Los documentos más similares pertenecen a las categorías:
rec.motorcycles
rec.motorcycles
rec.

conclusión:  

**2**. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación
(f1-score macro) en el conjunto de datos de test. Considerar cambiar parámteros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial
y ComplementNB.

La solución de este punto está basada en la página : https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_text_feature_extraction.html

## Usando complementNB

In [9]:
pipeline = Pipeline(
    [
        ("vect", TfidfVectorizer()),
        ("clf", ComplementNB()),
    ]
)
pipeline

Pipeline(steps=[('vect', TfidfVectorizer()), ('clf', ComplementNB())])

In [11]:
# parámentros para hacer el randomsearch o si se quiere el gridsearch
parameter_grid = {
    "vect__max_df": (0.2, 0.4, 0.6, 0.8, 1.0), # parametro del vectorizador para ignorar stop words o palabras cómunes en %X.
    "vect__min_df": (1, 3, 5, 10),# parámetro del vectorizador para ignorar palabras que aparecen en más de X documentos.
    "vect__ngram_range": ((1, 1), (1, 2)),  # unigramas or bigramas.
    "vect__norm": ("l1", "l2"),# tipo de normalización de los vectores.
    "clf__alpha": np.logspace(-6, 6, 13), # parámetros del clasificador que eligé el suavizado.
}

El f1_macro es especialmente útil cuando se tienen clases desbalanceadas
y se quiere un balance entre precisión y recall para cada clase. 

In [12]:
# se hace la búsqueda de los mejores hiperparámentros con f1_macro
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=parameter_grid,
    n_iter=40, # se puede aumentar el número de experimentos.
    random_state=0,
    n_jobs=2,# 2 procesos en paralelo 
    verbose=1,
    scoring='f1_macro'
)

pprint(parameter_grid)

{'clf__alpha': array([1.e-06, 1.e-05, 1.e-04, 1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01,
       1.e+02, 1.e+03, 1.e+04, 1.e+05, 1.e+06]),
 'vect__max_df': (0.2, 0.4, 0.6, 0.8, 1.0),
 'vect__min_df': (1, 3, 5, 10),
 'vect__ngram_range': ((1, 1), (1, 2)),
 'vect__norm': ('l1', 'l2')}


In [13]:
# se hace el fit de la búsqueda de los hiperparámetros 
random_search.fit(newsgroups_train.data, newsgroups_train.target)


Fitting 5 folds for each of 40 candidates, totalling 200 fits


/Users/tatianaarenas/Documents/ML_despliegue/procesamiento_lenguaje_natural/.venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


RandomizedSearchCV(estimator=Pipeline(steps=[('vect', TfidfVectorizer()),
                                             ('clf', ComplementNB())]),
                   n_iter=40, n_jobs=2,
                   param_distributions={'clf__alpha': array([1.e-06, 1.e-05, 1.e-04, 1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01,
       1.e+02, 1.e+03, 1.e+04, 1.e+05, 1.e+06]),
                                        'vect__max_df': (0.2, 0.4, 0.6, 0.8,
                                                         1.0),
                                        'vect__min_df': (1, 3, 5, 10),
                                        'vect__ngram_range': ((1, 1), (1, 2)),
                                        'vect__norm': ('l1', 'l2')},
                   random_state=0, scoring='f1_macro', verbose=1)

In [15]:
print("Los mejores parámetros encontrados fueron:")
best_parameters = random_search.best_estimator_.get_params()
for param_name in sorted(parameter_grid.keys()):
    print(f"{param_name}: {best_parameters[param_name]}")

Los mejores parámetros encontrados fueron:
clf__alpha: 0.1
vect__max_df: 0.2
vect__min_df: 1
vect__ngram_range: (1, 2)
vect__norm: l2


In [18]:
test_f1_macro = random_search.score(newsgroups_train.data, newsgroups_train.target)
print(
    " El mejor F1_macro usando  mejores parámetros"
    f" del random search es: {random_search.best_score_:.3f}"
)
print(f"F1 macro aplicado a conjunto de test fue: {test_f1_macro:.3f}")

 El mejor F1_macro usando  mejores parámetros del random search es: 0.772
F1 macro aplicado a conjunto de test fue: 0.972


## Usando el MultinomialNB

In [29]:
pipeline_multi = Pipeline(
    [
        ("vect", TfidfVectorizer()),
        ("clf", MultinomialNB()),
    ]
)
pipeline_multi

Pipeline(steps=[('vect', TfidfVectorizer()), ('clf', MultinomialNB())])

In [30]:
# usare el mismo param_grid que use para el complementNB
random_search_multi= RandomizedSearchCV(
    estimator=pipeline_multi,
    param_distributions=parameter_grid,
    n_iter=40,
    random_state=0,
    n_jobs=2,
    verbose=1,
    scoring='f1_macro'
)

In [32]:
random_search_multi.fit(newsgroups_train.data, newsgroups_train.target)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


/Users/tatianaarenas/Documents/ML_despliegue/procesamiento_lenguaje_natural/.venv/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


RandomizedSearchCV(estimator=Pipeline(steps=[('vect', TfidfVectorizer()),
                                             ('clf', MultinomialNB())]),
                   n_iter=40, n_jobs=2,
                   param_distributions={'clf__alpha': array([1.e-06, 1.e-05, 1.e-04, 1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01,
       1.e+02, 1.e+03, 1.e+04, 1.e+05, 1.e+06]),
                                        'vect__max_df': (0.2, 0.4, 0.6, 0.8,
                                                         1.0),
                                        'vect__min_df': (1, 3, 5, 10),
                                        'vect__ngram_range': ((1, 1), (1, 2)),
                                        'vect__norm': ('l1', 'l2')},
                   random_state=0, scoring='f1_macro', verbose=1)

In [27]:
print("Los mejores parámetros encontrados fueron:")
best_parameters_multi = random_search_multi.best_estimator_.get_params()
for param_name in sorted(parameter_grid.keys()):
    print(f"{param_name}: {best_parameters_multi[param_name]}")

Los mejores parámetros encontrados fueron:
clf__alpha: 0.01
vect__max_df: 0.2
vect__min_df: 5
vect__ngram_range: (1, 2)
vect__norm: l2


In [28]:
test_f1_macro_multi = random_search_multi.score(newsgroups_train.data, newsgroups_train.target)
print(
    " El mejor F1_macro usando  mejores parámetros"
    f" del random search es: {random_search_multi.best_score_:.3f}"
)
print(f"F1 macro aplicado a conjunto de test fue: {test_f1_macro_multi:.3f}")

 El mejor F1_macro usando  mejores parámetros del random search es: 0.720
F1 macro aplicado a conjunto de test fue: 0.969


**3**. Transponer la matriz documento-término. De esa manera se obtiene una matriz
término-documento que puede ser interpretada como una colección de vectorización de palabras.
Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares. **La elección de palabras no debe ser al azar para evitar la aparición de términos poco interpretables, elegirlas "manualmente"**.

In [35]:
# Transponemos la matriz para obtener una matriz término-documento
X_train_T = X_train.T
# escojo set de palabras "interpretables"
words = ["rationalist", "freethinker", "citizens","binoculars", "blender" ]

In [43]:
for word in words:

    word_idx = tfidfvect.vocabulary_[word]  # Busco el índice de la palabra

    # Obtengo el vector de la palabra como matriz bidimensional
    word_vector = X_train_T[word_idx].toarray().reshape(1, -1)  

    # Calculo la similitud del coseno entre la palabra  y todas las demás
    similaridad_coseno = cosine_similarity(word_vector, X_train_T)
    indices_similares = np.argsort(similaridad_coseno[0])[::-1]
    similaridad_coseno[0][indices_similares]

    # Muestro las 5 palabras más similares a 'engineering'
    mas_similares = [idx2word[idx] for idx in indices_similares[1:6]]

    print(f"Para la palabra {word} las 5 mas_similares son: {mas_similares}")    

Para la palabra rationalist las 5 mas_similares son: ['adulteries', '8018', 'islington', '910309', '14215']
Para la palabra freethinker las 5 mas_similares son: ['adulteries', '8018', 'islington', '910309', '14215']
Para la palabra citizens las 5 mas_similares son: ['pencils', 'cruptology', 'fmgs', 'felony', 'stron']
Para la palabra binoculars las 5 mas_similares son: ['monolux', 'matic', 'bro', '7x35', 'duster']
Para la palabra blender las 5 mas_similares son: ['duster', 'osterizer', 'vaccum', 'monolux', '7x35']
